# @wrap_model_call——模型调用拦截

`@wrap_model_call` 不像 before/after 那样只是观察，而是可以完全控制模型执行——重试、降级、缓存、甚至跳过模型用预设回复。

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage

print("导入完成")

导入完成


## 理解 handler 回调

调用 `handler(request)` 才会真正执行模型；不调用则跳过模型。

In [2]:
@wrap_model_call
def my_middleware(request, handler):
    """演示 wrap_model_call 的基本结构"""
    print("模型即将被调用...")
    response = handler(request)
    print("模型调用完成")
    return response

print("wrap_model_call 基本结构：handler(request) 执行模型")

wrap_model_call 基本结构：handler(request) 执行模型


## 场景 1：重试机制

In [4]:
@wrap_model_call
def retry_on_error(request, handler):
    """模型调用失败时自动重试，最多 3 次"""
    max_retries = 3
    last_error = None
    for attempt in range(max_retries):
        try:
            result = handler(request)
            if attempt > 0:
                print(f" [重试成功] 第 {attempt+1} 次尝试")
            return result
        except Exception as e:
            last_error = e
            if attempt < max_retries - 1:
                print(f" [重试] 第 {attempt+1} 次失败，重试...")
                import time
                time.sleep((attempt + 1) * 2)
    raise last_error

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[retry_on_error], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

result = agent.invoke({"messages": [HumanMessage(content="介绍一下菜鸟教程")]})
print(f"\n回复: {result['messages'][-1].content[:80]}...")


回复: 菜鸟教程（RUNOOB.COM）是一个面向编程初学者的中文技术学习平台，提供免费、系统的入门教程，涵盖 HTML、CSS、JavaScript、Python、J...


## 场景 2：模型降级/故障转移

In [5]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import AIMessage

primary_model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
fallback_model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0, max_tokens=100)

@wrap_model_call
def fallback_on_error(request, handler):
    """主模型失败时切换到备用模型"""
    try:
        return handler(request)
    except Exception as e:
        print(f"[降级] 主模型失败，切换到备用模型...")
        request = request.override(model=fallback_model)
        try:
            return handler(request)
        except Exception:
            return AIMessage(content="抱歉，服务暂时不可用，请稍后重试。")

print("故障转移中间件已定义")

故障转移中间件已定义


## 场景 3：缓存模型响应

In [8]:
from langchain.agents.middleware import wrap_model_call
from langchain.messages import AIMessage

cache = {}

@wrap_model_call
def cache_responses(request, handler):
    """缓存模型响应（当前版本 handler 返回 ModelResponse）"""
    messages = request.messages
    if not messages:
        return handler(request)
    last_content = str(messages[-1].content) if hasattr(messages[-1], "content") else ""
    cache_key = last_content[:200]
    if cache_key in cache:
        print("[缓存命中] 直接返回")
        return AIMessage(content=f"{cache[cache_key]}\n\n*(来自缓存)*")
    result = handler(request)
    # ModelResponse.result 是 list[AIMessage]
    if hasattr(result, "result") and result.result:
        cache[cache_key] = result.result[0].content
        print(f"[缓存写入] 当前 {len(cache)} 条")
    return result

model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(model=model, middleware=[cache_responses], system_prompt="你是菜鸟教程 RUNOOB 的助手。")

result1 = agent.invoke({"messages": [{"role": "user", "content": "Python 是什么？"}]})
result2 = agent.invoke({"messages": [{"role": "user", "content": "Python 是什么？"}]})


[缓存写入] 当前 1 条
[缓存命中] 直接返回


## 场景 4：修改 request——动态注入系统消息

In [9]:
from datetime import datetime
from langchain.agents.middleware import wrap_model_call
from langchain.messages import SystemMessage

@wrap_model_call
def inject_time_context(request, handler):
    """在每次模型调用前注入当前时间"""
    now = datetime.now()
    time_context = f"当前时间：{now.strftime('%Y年%m月%d日 %H:%M')}。"
    if request.system_message:
        new_content = f"{request.system_message.content}\n\n{time_context}"
    else:
        new_content = time_context
    new_request = request.override(system_message=SystemMessage(content=new_content))
    return handler(new_request)

print("时间注入中间件已定义")

时间注入中间件已定义


## 场景 5：多个 wrap_model_call 的组合

多个中间件像洋葱一样层层包裹。最外层最先执行、最后返回。

In [10]:
@wrap_model_call
def outer_middleware(request, handler):
    """最外层中间件"""
    print("[外层] 开始")
    result = handler(request)
    print("[外层] 结束")
    return result

@wrap_model_call
def inner_middleware(request, handler):
    """内层中间件"""
    print(" [内层] 开始")
    result = handler(request)
    print(" [内层] 结束")
    return result

print("多个 wrap_model_call 会按洋葱模型组合")

多个 wrap_model_call 会按洋葱模型组合


**术语：**

- **@wrap_model_call**：包裹模型调用的中间件，可完全控制模型执行
- **handler(request)**：真正执行模型的回调
- **request.override()**：创建新的 request 副本（不可变）